# 🏋️ Azure AI Search + Microsoft Agent Framework: Fitness-Fun Workshop 🤸

Welcome to this self-guided workshop where you'll:

1. **Create** an Azure AI Search index containing sample fitness equipment data
2. **Upload** and verify your documents
3. **Create** a Fitness Shopping Agent using Microsoft's `ChatCompletionsClient`
4. **Run** conversational queries to search and recommend items from your index

> **Note:** This demo uses **Microsoft Azure AI frameworks** directly via the `azure-ai-projects`, `azure-ai-inference`, and `azure-search-documents` SDKs — no third-party agent frameworks required.
>
> ```bash
> pip install azure-ai-projects azure-identity azure-identity-broker azure-search-documents azure-ai-inference python-dotenv
> ```

Ensure you've set the following environment variables in your `.env` file:

- `PROJECT_ENDPOINT` (e.g. `https://{hub}.services.ai.azure.com/api/projects/{project}`)
- `AZURE_OPENAI_KEY` (API key for Azure OpenAI inference)
- `MODEL_DEPLOYMENT_NAME` (e.g. `gpt-4o`)

Let's get started!

## Prerequisites

Before running the cells below, please verify:

1. You have installed the required Microsoft dependencies:
   ```bash
   pip install azure-ai-projects azure-identity azure-identity-broker azure-search-documents azure-ai-inference python-dotenv
   ```

2. Your `.env` file at the workspace root contains:
   - `PROJECT_ENDPOINT` — e.g. `https://{hub}.services.ai.azure.com/api/projects/{project}`
   - `AZURE_OPENAI_KEY` — API key for Azure OpenAI inference
   - `MODEL_DEPLOYMENT_NAME` — e.g. `gpt-5.4`

3. You are signed in to Azure (via VS Code Azure extension, `az login`, or `azd auth login`). The notebook uses `DefaultAzureCredential` to authenticate automatically.

4. Your Azure AI Foundry project has an **Azure AI Search** connection configured.


## 1. Create & Populate Azure AI Search Index

In this section we will:

1. **Create** an Azure AI Search index called `myfitnessindex` with a schema suited for fitness items
2. **Upload** sample documents containing fitness equipment data
3. **Verify** that the documents are searchable

Make sure your environment has the appropriate search credentials (typically obtained via your AI Foundry project).

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from azure.ai.projects import AIProjectClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.search.documents import SearchClient

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

project_endpoint = os.getenv("PROJECT_ENDPOINT", "")
if not project_endpoint:
    raise ValueError("PROJECT_ENDPOINT not set. Please add it to your .env file.")

# Derive base endpoint (scheme + host) for inference calls in later cells
_parsed       = urlparse(project_endpoint)
base_endpoint = f"{_parsed.scheme}://{_parsed.netloc}"

index_name      = "myfitnessindex"
use_mock_search = False

# ── Initialize AIProjectClient ─────────────────────────────────────────────────
try:
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=DefaultAzureCredential(),
    )
    print("✅ Initialized AIProjectClient")

    # Get the default Azure AI Search connection (with credentials)
    # Use 'CognitiveSearch' string — the enum value in 2.x cannot be passed as keyword arg
    search_conn  = project_client.connections.get_default("CognitiveSearch", include_credentials=True)
    _search_key  = search_conn.credentials.get("key") or search_conn.credentials["key"]
    _search_cred = AzureKeyCredential(_search_key)

    index_client = SearchIndexClient(
        endpoint=search_conn.target,
        credential=_search_cred,
    )
    print("✅ Created SearchIndexClient")

    search_client = SearchClient(
        endpoint=search_conn.target,
        index_name=index_name,
        credential=_search_cred,
    )
    print("✅ Created SearchClient for document operations")

except Exception as e:
    print(f"⚠️  Could not connect to Azure AI project: {e}")
    print("⚠️  Falling back to mock search clients for demonstration.")
    use_mock_search = True

# ── Mock clients (fallback when Azure credentials are unavailable) ─────────────
if use_mock_search:
    class MockSearchClient:
        def __init__(self):
            self.documents = []

        def search(self, search_text=None, filter=None, top=10):
            if not search_text:
                return self.documents[:top]
            keywords = search_text.lower().split()
            return [
                doc for doc in self.documents
                if any(kw in (doc["Name"] + " " + doc["Category"] + " " + doc["Description"]).lower()
                       for kw in keywords)
            ][:top]

        def upload_documents(self, documents):
            self.documents = documents
            return f"Uploaded {len(documents)} documents"

    class MockIndexClient:
        def list_indexes(self):
            return [type('Index', (), {'name': index_name})()]
        def delete_index(self, name):
            return None
        def create_index(self, index):
            return type('Index', (), {'name': index.name})()

    search_client = MockSearchClient()
    index_client  = MockIndexClient()
    print("✅ Using Mock SearchClient (demonstration mode)")
    print("✅ Using Mock SearchIndexClient (demonstration mode)")


### Define the Index Schema

We will create an index with the following fields:

- `FitnessItemID`: Unique key
- `Name`: Searchable text field (also filterable)
- `Category`: Searchable, filterable, and facetable (e.g. Strength, Cardio, Flexibility)
- `Price`: Numeric field (filterable, sortable, and facetable)
- `Description`: Full-text searchable field

In [ ]:
def create_fitness_index():
    """Create fitness index (works with both real and mock search clients)."""
    if not use_mock_search:
        # Real index creation
        fields = [
            SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
            SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
            SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
            SearchableField(name="Description", type=SearchFieldDataType.String)
        ]
        
        index = SearchIndex(name=index_name, fields=fields)
        
        # Delete the index if it already exists (for a fresh start)
        if index_name in [x.name for x in index_client.list_indexes()]:
            index_client.delete_index(index_name)
            print(f"🗑️ Deleted existing index: {index_name}")
        
        created = index_client.create_index(index)
        print(f"🎉 Created index: {created.name}")
    else:
        # Mock index creation
        print(f"🎉 Mock index created: {index_name}")

# Create the index
create_fitness_index()

### Upload Sample Documents

Now we’ll add some sample fitness items to `myfitnessindex`.

In [ ]:
def upload_fitness_docs():
    sample_docs = [
        {
            "FitnessItemID": "1",
            "Name": "Adjustable Dumbbell",
            "Category": "Strength",
            "Price": 59.99,
            "Description": "A compact, adjustable weight for targeted muscle workouts."
        },
        {
            "FitnessItemID": "2",
            "Name": "Yoga Mat",
            "Category": "Flexibility",
            "Price": 25.0,
            "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
        },
        {
            "FitnessItemID": "3",
            "Name": "Treadmill",
            "Category": "Cardio",
            "Price": 499.0,
            "Description": "A sturdy treadmill with adjustable speed and incline settings."
        },
        {
            "FitnessItemID": "4",
            "Name": "Resistance Bands",
            "Category": "Strength",
            "Price": 15.0,
            "Description": "Set of colorful bands for light to moderate resistance workouts."
        }
    ]

    result = search_client.upload_documents(documents=sample_docs)
    # Convert result objects to dictionaries for proper display
    result_list = [
        {
            "key": r.key,
            "status": r.status_code in (200, 201),
            "errorMessage": r.error_message,
            "statusCode": r.status_code
        }
        for r in result
    ]
    print(f"🚀 Upload result: {result_list}")

upload_fitness_docs()
print("✅ Documents uploaded to search index")


### Verify the Documents

Let’s perform a basic search query (e.g. for items in the **Strength** category) to ensure everything is working.

In [ ]:
results = search_client.search(search_text="Strength", filter=None, top=10)

print("🔍 Search results for 'Strength':")
print("-" * 50)
found_items = False
for doc in results:
    found_items = True
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 50)

if not found_items:
    print("No matching items found.")

## 2. Create Fitness Shopping Agent with Azure AI Search (Microsoft Framework)

In this section we'll create a **simple, synchronous fitness shopping agent** using Microsoft's `azure.ai.inference.ChatCompletionsClient` and `azure-search-documents` SDK. This agent:

- Uses your deployed model (specified by `MODEL_DEPLOYMENT_NAME`, e.g. `gpt-5.4`)
- Queries `myfitnessindex` in real-time for fitness recommendations
- Maintains conversation history for multi-turn interactions
- Uses **no third-party frameworks** — just Microsoft's native SDKs

The code is brief and easy to debug.

In [ ]:
import json
import re
from dataclasses import dataclass, field
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage, AssistantMessage
from azure.core.credentials import AzureKeyCredential

# ── Setup inference client ─────────────────────────────────────────────────────
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")
api_key               = os.getenv("AZURE_OPENAI_KEY", "")
_deploy_endpoint      = f"{base_endpoint}/openai/deployments/{model_deployment_name}"

use_mock_inference = not bool(api_key.strip())
inference_client   = None

if not use_mock_inference:
    try:
        inference_client = ChatCompletionsClient(
            endpoint=_deploy_endpoint,
            credential=AzureKeyCredential(api_key),
        )
        print("✅ ChatCompletionsClient ready")
    except Exception as e:
        print(f"⚠️  Could not create ChatCompletionsClient: {e}")
        use_mock_inference = True
else:
    print("⚠️  AZURE_OPENAI_KEY not set — using mock inference for demonstration.")

# ── FitnessShoppingAgent ───────────────────────────────────────────────────────
@dataclass
class FitnessShoppingAgent:
    """Synchronous fitness shopping assistant using Microsoft ChatCompletionsClient."""
    name: str
    search_client: object
    model_name: str
    inference_client: object = None
    use_mock: bool = False
    history: list = field(default_factory=list)

    def _get_all_items(self) -> list:
        """Fetch all items from the search index."""
        try:
            return [
                {
                    "name": d["Name"],
                    "category": d["Category"],
                    "price": d["Price"],
                    "description": d["Description"],
                }
                for d in self.search_client.search(search_text="", top=100)
            ]
        except Exception as e:
            print(f"Search error: {e}")
            return []

    def query(self, user_message: str) -> str:
        """Process a user query and return the agent response."""
        self.history.append({"role": "user", "content": user_message})
        all_items = self._get_all_items()

        if self.use_mock:
            lower = user_message.lower()

            # Category filter
            if any(w in lower for w in ("strength", "weight", "dumbbell")):
                items = [i for i in all_items if i["category"].lower() == "strength"]
            elif any(w in lower for w in ("cardio", "run", "treadmill")):
                items = [i for i in all_items if i["category"].lower() == "cardio"]
            elif any(w in lower for w in ("flex", "yoga")):
                items = [i for i in all_items if i["category"].lower() == "flexibility"]
            else:
                items = all_items

            # Budget filter
            budget_match = re.search(r'\$?(\d+)', user_message)
            if ("under" in lower or "$" in user_message) and budget_match:
                budget = float(budget_match.group(1))
                items = [i for i in items if i["price"] <= budget]

            if items:
                answer = "Recommended items:\n\n"
                for item in items:
                    answer += f"• **{item['name']}** — ${item['price']:.2f}\n"
                    answer += f"  Category: {item['category']}\n"
                    answer += f"  {item['description']}\n\n"
            else:
                answer = "No items found matching your criteria in our inventory."

        else:
            system_prompt = (
                "You are a Fitness Shopping Assistant. Recommend items ONLY from the inventory below.\n"
                "Rules:\n"
                "1. Filter by category if the user specifies one (cardio/strength/flexibility).\n"
                "2. Filter by budget if the user mentions a price limit.\n"
                "3. Include product name, price, category, and description.\n"
                "4. If nothing matches, say so clearly."
            )
            messages = [
                SystemMessage(content=system_prompt),
                UserMessage(
                    content=(
                        f"INVENTORY:\n{json.dumps(all_items, indent=2)}\n\n"
                        f"REQUEST: {user_message}"
                    )
                ),
            ]
            response = self.inference_client.complete(model=self.model_name, messages=messages)
            answer = response.choices[0].message.content

        self.history.append({"role": "assistant", "content": answer})
        return answer


# ── Create and run the agent ───────────────────────────────────────────────────
agent = FitnessShoppingAgent(
    name="FitnessShoppingAssistant",
    search_client=search_client,
    model_name=model_deployment_name,
    inference_client=inference_client,
    use_mock=use_mock_inference,
)
mode = "Microsoft ChatCompletionsClient" if not use_mock_inference else "mock inference (demonstration mode)"
print(f"✅ Created {agent.name} using {mode}")

print("\n" + "=" * 60)
print("Fitness Shopping Assistant Demo")
print("=" * 60)

for q in [
    "Which items are best for strength training?",
    "I need something for cardio under $300. Any suggestions?",
]:
    print(f"\n👤 User: {q}")
    print(f"\n🤖 Agent: {agent.query(q)}")
    print("-" * 60)

print("\n✅ Demo complete!")


## 3. Cleanup

Run the cell below to delete the `myfitnessindex` search index and start fresh on the next run.

In [ ]:
try:
    index_client.delete_index(index_name)
    print(f"🗑️ Deleted index {index_name}")
except Exception as e:
    print(f"Error deleting index: {e}")

# 🎉 Congrats!

You've successfully:

1. Created an Azure AI Search index and populated it with fitness data
2. Verified the data via a basic search query
3. Built and run a Microsoft Agent Framework agent that leverages Azure AI Search to answer natural language queries

Feel free to explore further enhancements (e.g. integrating more tools or advanced evaluation) and enjoy your journey with Azure AI Foundry and Microsoft Agent Framework!